# Adapter Smoke Test

Directly load a trained continual SD-LoRA adapter on top of the DINO backbone, inspect its metadata, run one prediction, and optionally sanity-check a small folder of images.

This notebook now searches Drive for adapter bundles and lets you choose which one to load.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'src').is_dir() and (candidate / 'scripts').is_dir():
        ROOT = candidate
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    from scripts.colab_repo_bootstrap import install_colab_requirements, mount_drive_if_available, resolve_repo_root, running_in_colab
except ModuleNotFoundError:
    clone_target = Path('/content/bitirmeprojesi')
    repo_url = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
    if not (clone_target / 'scripts').exists():
        subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(clone_target)], check=True)
    os.chdir(clone_target)
    if str(clone_target) not in sys.path:
        sys.path.insert(0, str(clone_target))
    from scripts.colab_repo_bootstrap import install_colab_requirements, mount_drive_if_available, resolve_repo_root, running_in_colab

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if running_in_colab():
    mount_drive_if_available()
    install_colab_requirements(ROOT / 'requirements_colab.txt', in_colab=True)

print(ROOT)

In [ ]:
import json
from collections import Counter
from pathlib import Path

from IPython.display import display

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import ipywidgets as widgets
except Exception:
    widgets = None

from scripts.colab_adapter_smoke_test import (
    discover_adapter_candidates,
    load_adapter_summary,
    predict_image_folder,
    predict_single_image,
)

CROP_NAME = None
# Optional crop filter, e.g. 'tomato'. Leave as None to show all discovered adapters.

ADAPTER_DIR = None
# Optional manual override. If set, discovery is skipped and this adapter is used directly.

ADAPTER_ROOT = None
# Fallback deployed root if you do not want to use discovery.

DRIVE_SEARCH_ROOTS = [
    Path('/content/drive/MyDrive/aads_ulora'),
    ROOT / 'outputs',
]
SELECTED_ADAPTER_INDEX = 0
IMAGE_PATH = None
BATCH_IMAGE_DIR = None
DEVICE = 'cuda'

print('Manual ADAPTER_DIR overrides discovery. Otherwise the notebook searches DRIVE_SEARCH_ROOTS and prompts for a selection.')

In [ ]:
adapter_candidates = []
adapter_selector = None

if ADAPTER_DIR is not None:
    SELECTED_ADAPTER_DIR = str(Path(ADAPTER_DIR))
    print(f'Using manual ADAPTER_DIR: {SELECTED_ADAPTER_DIR}')
else:
    adapter_candidates = discover_adapter_candidates(DRIVE_SEARCH_ROOTS, crop_name=CROP_NAME)
    if not adapter_candidates:
        raise FileNotFoundError(
            'No adapters found under DRIVE_SEARCH_ROOTS. Set ADAPTER_DIR manually or update DRIVE_SEARCH_ROOTS.'
        )

    print('Discovered adapters:')
    for index, candidate in enumerate(adapter_candidates):
        print(f'[{index}] {candidate["display_name"]}')

    if widgets is not None:
        adapter_selector = widgets.Dropdown(
            options=[(candidate['display_name'], index) for index, candidate in enumerate(adapter_candidates)],
            value=min(SELECTED_ADAPTER_INDEX, len(adapter_candidates) - 1),
            description='Adapter:',
            layout=widgets.Layout(width='95%'),
        )
        display(adapter_selector)
        print('Choose an adapter in the dropdown, then run the next cell.')
    else:
        print('ipywidgets unavailable. Set SELECTED_ADAPTER_INDEX to the adapter you want, then run the next cell.')

In [ ]:
if ADAPTER_DIR is None:
    selected_index = adapter_selector.value if adapter_selector is not None else SELECTED_ADAPTER_INDEX
    selected_candidate = adapter_candidates[int(selected_index)]
    ADAPTER_DIR = selected_candidate['adapter_dir']
    if CROP_NAME is None:
        CROP_NAME = selected_candidate.get('crop_name')
    print(json.dumps(selected_candidate, indent=2))

if CROP_NAME is None:
    raise ValueError('Set CROP_NAME manually or select an adapter candidate with crop metadata.')

summary = load_adapter_summary(
    CROP_NAME,
    adapter_dir=ADAPTER_DIR,
    adapter_root=ADAPTER_ROOT,
    config_env='colab',
    device=DEVICE,
)

print(json.dumps(summary, indent=2))

In [ ]:
if IMAGE_PATH is None:
    if not running_in_colab():
        raise ValueError('Set IMAGE_PATH when running outside Colab.')
    from google.colab import files
    uploaded = files.upload()
    IMAGE_PATH = next(iter(uploaded.keys()))

single_result = predict_single_image(
    IMAGE_PATH,
    CROP_NAME,
    adapter_dir=ADAPTER_DIR,
    adapter_root=ADAPTER_ROOT,
    config_env='colab',
    device=DEVICE,
)

print(json.dumps(single_result, indent=2))

In [ ]:
if not BATCH_IMAGE_DIR:
    print('Set BATCH_IMAGE_DIR to run the optional folder sanity check.')
else:
    rows = predict_image_folder(
        BATCH_IMAGE_DIR,
        CROP_NAME,
        adapter_dir=ADAPTER_DIR,
        adapter_root=ADAPTER_ROOT,
        config_env='colab',
        device=DEVICE,
    )
    if pd is not None:
        df = pd.DataFrame(rows)
        display(df)
    else:
        print(rows)

    predicted_counts = Counter(row['predicted_class'] for row in rows if row.get('predicted_class'))
    ood_count = sum(1 for row in rows if row.get('is_ood') is True)
    error_count = sum(1 for row in rows if row.get('error'))

    print('predicted_class_counts:', dict(predicted_counts))
    print('ood_count:', ood_count)
    print('error_count:', error_count)